In [3]:
import warnings
from pathlib import Path

import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from tqdm import tqdm

# ============================================================
# Disable RDKit logging
# ============================================================
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

# ============================================================
# 1. File paths and column names
# ============================================================
train_csv_path = (
    "../../dataset_tadf/dataset_gen/"
    "gendata_est_sa/token_dataset/train.csv"
)

train_smiles_col = "TADF_SMILES"
generated_smiles_col = "SMILES"

generated_files = [
    'gen_smiles_EST_0.01_SA_6_T1.0_epoch049.csv',
    'gen_smiles_EST_0.2_SA_1_T1.0_epoch049.csv',
    'gen_smiles_EST_0.05_SA_2.5_T1.0_epoch049.csv',
    'gen_smiles_EST_0.05_SA_3.5_T1.0_epoch049.csv',
    'gen_smiles_EST_0.05_SA_None_T1.0_epoch049.csv',
    'gen_smiles_EST_0.15_SA_2.5_T1.0_epoch049.csv',
    'gen_smiles_EST_0.15_SA_3.5_T1.0_epoch049.csv',
    'gen_smiles_EST_0.15_SA_None_T1.0_epoch049.csv',
    'gen_smiles_EST_None_SA_2.5_T1.0_epoch049.csv',
    'gen_smiles_EST_None_SA_3.5_T1.0_epoch049.csv',
    'gen_smiles_EST_None_SA_None_T1.0_epoch049.csv',
]


# ============================================================
# 2. SMILES canonicalization
# ============================================================
def canonicalize_smiles(smiles):
    """
    Convert SMILES to canonical SMILES.
    Return None for invalid, empty, or malformed SMILES.
    """
    if pd.isna(smiles):
        return None

    smiles = str(smiles).strip()

    if not smiles:
        return None

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True
    )


# ============================================================
# 3. Extract standard Bemis–Murcko scaffolds
# ============================================================
def get_bemis_murcko_scaffold(canonical_smiles):
    """
    Extract a standard Bemis–Murcko scaffold retaining atom and bond types.
    Return None for invalid or acyclic molecules.
    """
    mol = Chem.MolFromSmiles(canonical_smiles)

    if mol is None:
        return None

    scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)

    if scaffold_mol is None or scaffold_mol.GetNumAtoms() == 0:
        return None

    scaffold_smiles = Chem.MolToSmiles(
        scaffold_mol,
        canonical=True,
        isomericSmiles=False
    )

    return scaffold_smiles if scaffold_smiles else None


# ============================================================
# 4. Extract conditions from filenames
# ============================================================
def extract_condition(file_name):
    """
    Extract EST and SA conditions from the filename.
    """
    stem = Path(file_name).stem

    est_part = stem.split("_EST_", 1)[1].split("_SA_", 1)[0]
    sa_part = stem.split("_SA_", 1)[1].split("_T1.0_", 1)[0]

    return est_part, sa_part


# ============================================================
# 5. Read and process the original training set once
# ============================================================
print("=" * 100)
print("Reading the original training set...")

df_train = pd.read_csv(train_csv_path)

if train_smiles_col not in df_train.columns:
    raise ValueError(
        f"Missing training-set column: {train_smiles_col}\n"
        f"Available columns: {df_train.columns.tolist()}"
    )

train_canonical_set = set()
train_invalid_count = 0

for smiles in tqdm(
    df_train[train_smiles_col].tolist(),
    desc="Canonicalizing training molecules"
):
    canonical_smiles = canonicalize_smiles(smiles)

    if canonical_smiles is None:
        train_invalid_count += 1
        continue

    train_canonical_set.add(canonical_smiles)

# Extract unique training-set scaffolds
train_scaffold_set = set()
train_acyclic_count = 0

for canonical_smiles in tqdm(
    train_canonical_set,
    desc="Extracting training-set Bemis–Murcko scaffolds"
):
    scaffold = get_bemis_murcko_scaffold(canonical_smiles)

    if scaffold is None:
        train_acyclic_count += 1
        continue

    train_scaffold_set.add(scaffold)

print(f"Raw training records: {len(df_train):,}")
print(f"Invalid training records: {train_invalid_count:,}")
print(f"Valid unique training molecules: {len(train_canonical_set):,}")
print(f"Unique training-set Bemis–Murcko scaffolds: {len(train_scaffold_set):,}")
print(f"Training molecules with acyclic or empty scaffolds: {train_acyclic_count:,}")


# ============================================================
# 6. Process the 10 generated files individually
# ============================================================
all_results = []

for file_path in generated_files:

    print("\n" + "=" * 100)
    print(f"Processing: {file_path}")

    if not Path(file_path).exists():
        print("File not found. Skipping.")
        continue

    df_gen = pd.read_csv(file_path)

    if generated_smiles_col not in df_gen.columns:
        print(
            f"Missing column {generated_smiles_col}. Skipping.\n"
            f"Available columns: {df_gen.columns.tolist()}"
        )
        continue

    total_raw = len(df_gen)

    # Track all valid canonical molecules for within-file deduplication
    seen_valid_smiles = set()

    # Valid, unique, and novel (VUN) molecules retained after three filtering steps
    vun_smiles_set = set()

    invalid_count = 0
    duplicate_count = 0
    train_overlap_count = 0

    for smiles in tqdm(
        df_gen[generated_smiles_col].tolist(),
        desc="VUN filtering",
        leave=False
    ):
        # 1. Remove invalid or malformed SMILES
        canonical_smiles = canonicalize_smiles(smiles)

        if canonical_smiles is None:
            invalid_count += 1
            continue

        # 2. Remove duplicates within the generated file
        if canonical_smiles in seen_valid_smiles:
            duplicate_count += 1
            continue

        # Immediately add each first-seen valid molecule to the deduplication set
        seen_valid_smiles.add(canonical_smiles)

        # 3. Remove complete molecules already present in the original training set
        if canonical_smiles in train_canonical_set:
            train_overlap_count += 1
            continue

        vun_smiles_set.add(canonical_smiles)

    # ========================================================
    # 7. Extract Bemis–Murcko scaffolds from VUN molecules
    # ========================================================
    vun_scaffold_set = set()
    vun_acyclic_count = 0

    for canonical_smiles in tqdm(
        vun_smiles_set,
        desc="Extracting VUN scaffolds",
        leave=False
    ):
        scaffold = get_bemis_murcko_scaffold(canonical_smiles)

        if scaffold is None:
            vun_acyclic_count += 1
            continue

        vun_scaffold_set.add(scaffold)

    known_scaffold_set = (
        vun_scaffold_set & train_scaffold_set
    )

    novel_scaffold_set = (
        vun_scaffold_set - train_scaffold_set
    )

    num_vun_molecules = len(vun_smiles_set)
    num_unique_scaffolds = len(vun_scaffold_set)
    num_known_scaffolds = len(known_scaffold_set)
    num_novel_scaffolds = len(novel_scaffold_set)

    # Bemis–Murcko scaffold novelty
    if num_unique_scaffolds > 0:
        scaffold_novelty = (
            num_novel_scaffolds
            / num_unique_scaffolds
        )
    else:
        scaffold_novelty = 0.0

    # Scaffold diversity
    if num_vun_molecules > 0:
        scaffold_diversity = (
            num_unique_scaffolds
            / num_vun_molecules
        )
    else:
        scaffold_diversity = 0.0

    est_condition, sa_condition = extract_condition(file_path)

    result = {
        "File": file_path,
        "Delta_EST": est_condition,
        "SA_score": sa_condition,
        "Raw_molecules": total_raw,
        "Invalid": invalid_count,
        "Internal_duplicates": duplicate_count,
        "Training_overlaps": train_overlap_count,
        "VUN_molecules": num_vun_molecules,
        "Acyclic_or_empty_scaffold": vun_acyclic_count,
        "Unique_scaffolds": num_unique_scaffolds,
        "Known_scaffolds": num_known_scaffolds,
        "Novel_scaffolds": num_novel_scaffolds,
        "Scaffold_novelty": scaffold_novelty,
        "Scaffold_diversity": scaffold_diversity
    }

    all_results.append(result)

    # ========================================================
    # 8. Print results for the current file
    # ========================================================
    print("-" * 100)
    print(f"Delta_EST condition: {est_condition}")
    print(f"SA condition: {sa_condition}")
    print(f"Raw generated molecules: {total_raw:,}")
    print(f"Invalid/malformed records removed: {invalid_count:,}")
    print(f"Internal duplicates removed: {duplicate_count:,}")
    print(f"Training-set overlaps removed: {train_overlap_count:,}")
    print(f"Final VUN molecules: {num_vun_molecules:,}")
    print(f"Molecules with acyclic or empty scaffolds: {vun_acyclic_count:,}")
    print(f"Unique Bemis–Murcko scaffolds: {num_unique_scaffolds:,}")
    print(f"Scaffolds present in the training set: {num_known_scaffolds:,}")
    print(f"Novel scaffolds absent from the training set: {num_novel_scaffolds:,}")
    print(
        "Bemis–Murcko scaffold novelty: "
        f"{scaffold_novelty:.4f} "
        f"({scaffold_novelty * 100:.2f}%)"
    )
    print(
        "Scaffold diversity: "
        f"{scaffold_diversity:.4f} "
        f"({scaffold_diversity * 100:.2f}%)"
    )


# ============================================================
# 9. Print the final summary
#    Display results only; do not save any files
# ============================================================
if all_results:
    df_results = pd.DataFrame(all_results)

    summary_columns = [
        "Delta_EST",
        "SA_score",
        "Raw_molecules",
        "Invalid",
        "Internal_duplicates",
        "Training_overlaps",
        "VUN_molecules",
        "Unique_scaffolds",
        "Known_scaffolds",
        "Novel_scaffolds",
        "Scaffold_novelty",
        "Scaffold_diversity"
    ]

    df_summary = df_results[summary_columns].copy()

    # Convert to percentages for inclusion in manuscript tables
    df_summary["Scaffold_novelty_percent"] = (
        df_summary["Scaffold_novelty"] * 100
    )

    df_summary["Scaffold_diversity_percent"] = (
        df_summary["Scaffold_diversity"] * 100
    )

    print("\n" + "=" * 150)
    print("Summary results for the 10 conditions")
    print("=" * 150)

    print(
        df_summary.to_string(
            index=False,
            formatters={
                "Scaffold_novelty":
                    lambda x: f"{x:.4f}",
                "Scaffold_diversity":
                    lambda x: f"{x:.4f}",
                "Scaffold_novelty_percent":
                    lambda x: f"{x:.2f}",
                "Scaffold_diversity_percent":
                    lambda x: f"{x:.2f}"
            }
        )
    )

    print("\nCalculation complete. No new files were saved.")
else:
    print("\nNo generated files were processed successfully.")

Reading the original training set...


Extracting training-set Bemis–Murcko scaffolds: 100%|████████████████████████████████| 3092/3092 [00:04<00:00, 744.72it/s]


Raw training records: 3,093
Invalid training records: 0
Valid unique training molecules: 3,092
Unique training-set Bemis–Murcko scaffolds: 2,502
Training molecules with acyclic or empty scaffolds: 0

Processing: gen_smiles_EST_0.01_SA_6_T1.0_epoch049.csv


----------------------------------------------------------------------------------------------------
Delta_EST condition: 0.01
SA condition: 6
Raw generated molecules: 10,000
Invalid/malformed records removed: 3,436
Internal duplicates removed: 2,249
Training-set overlaps removed: 396
Final VUN molecules: 3,919
Molecules with acyclic or empty scaffolds: 0
Unique Bemis–Murcko scaffolds: 3,403
Scaffolds present in the training set: 217
Novel scaffolds absent from the training set: 3,186
Bemis–Murcko scaffold novelty: 0.9362 (93.62%)
Scaffold diversity: 0.8683 (86.83%)

Processing: gen_smiles_EST_0.2_SA_1_T1.0_epoch049.csv


----------------------------------------------------------------------------------------------------
Delta_EST condition: 0.2
SA condition: 1
Raw generated molecules: 10,000
Invalid/malformed records removed: 136
Internal duplicates removed: 9,126
Training-set overlaps removed: 36
Final VUN molecules: 702
Molecules with acyclic or empty scaffolds: 2
Unique Bemis–Murcko scaffolds: 263
Scaffolds present in the training set: 31
Novel scaffolds absent from the training set: 232
Bemis–Murcko scaffold novelty: 0.8821 (88.21%)
Scaffold diversity: 0.3746 (37.46%)

Processing: gen_smiles_EST_0.05_SA_2.5_T1.0_epoch049.csv


...

----------------------------------------------------------------------------------------------------
Delta_EST condition: 0.15
SA condition: 3.5
Raw generated molecules: 10,000
Invalid/malformed records removed: 1,668
Internal duplicates removed: 2,803
Training-set overlaps removed: 870
Final VUN molecules: 4,659
Molecules with acyclic or empty scaffolds: 0
Unique Bemis–Murcko scaffolds: 3,993
Scaffolds present in the training set: 368
Novel scaffolds absent from the training set: 3,625
Bemis–Murcko scaffold novelty: 0.9078 (90.78%)
Scaffold diversity: 0.8571 (85.71%)

Processing: gen_smiles_EST_0.15_SA_None_T1.0_epoch049.csv


----------------------------------------------------------------------------------------------------
Delta_EST condition: 0.15
SA condition: None
Raw generated molecules: 10,000
Invalid/malformed records removed: 1,725
Internal duplicates removed: 2,156
Training-set overlaps removed: 1,280
Final VUN molecules: 4,839
Molecules with acyclic or empty scaffolds: 0
Unique Bemis–Murcko scaffolds: 4,270
Scaffolds present in the training set: 444
Novel scaffolds absent from the training set: 3,826
Bemis–Murcko scaffold novelty: 0.8960 (89.60%)
Scaffold diversity: 0.8824 (88.24%)

Processing: gen_smiles_EST_None_SA_2.5_T1.0_epoch049.csv


----------------------------------------------------------------------------------------------------
Delta_EST condition: None
SA condition: 2.5
Raw generated molecules: 10,000
Invalid/malformed records removed: 530
Internal duplicates removed: 4,322
Training-set overlaps removed: 973
Final VUN molecules: 4,175
Molecules with acyclic or empty scaffolds: 0
Unique Bemis–Murcko scaffolds: 3,395
Scaffolds present in the training set: 339
Novel scaffolds absent from the training set: 3,056
Bemis–Murcko scaffold novelty: 0.9001 (90.01%)
Scaffold diversity: 0.8132 (81.32%)

Processing: gen_smiles_EST_None_SA_3.5_T1.0_epoch049.csv


----------------------------------------------------------------------------------------------------
Delta_EST condition: None
SA condition: 3.5
Raw generated molecules: 10,000
Invalid/malformed records removed: 1,571
Internal duplicates removed: 2,518
Training-set overlaps removed: 992
Final VUN molecules: 4,919
Molecules with acyclic or empty scaffolds: 0
Unique Bemis–Murcko scaffolds: 4,273
Scaffolds present in the training set: 416
Novel scaffolds absent from the training set: 3,857
Bemis–Murcko scaffold novelty: 0.9026 (90.26%)
Scaffold diversity: 0.8687 (86.87%)

Processing: gen_smiles_EST_None_SA_None_T1.0_epoch049.csv


----------------------------------------------------------------------------------------------------
Delta_EST condition: None
SA condition: None
Raw generated molecules: 10,000
Invalid/malformed records removed: 1,365
Internal duplicates removed: 2,124
Training-set overlaps removed: 1,414
Final VUN molecules: 5,097
Molecules with acyclic or empty scaffolds: 0
Unique Bemis–Murcko scaffolds: 4,485
Scaffolds present in the training set: 472
Novel scaffolds absent from the training set: 4,013
Bemis–Murcko scaffold novelty: 0.8948 (89.48%)
Scaffold diversity: 0.8799 (87.99%)

Summary results for the 10 conditions
Delta_EST SA_score  Raw_molecules  Invalid  Internal_duplicates  Training_overlaps  VUN_molecules  Unique_scaffolds  Known_scaffolds  Novel_scaffolds Scaffold_novelty Scaffold_diversity Scaffold_novelty_percent Scaffold_diversity_percent
     0.01        6          10000     3436                 2249                396           3919              3403              217             